In [ ]:
#%pip install -q pandas numpy matplotlib scikit-learn

In [11]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 100)
pd.set_option("display.precision", 4)

DATA_DIR = Path("data")
DATA_DIR.mkdir(exist_ok=True)

RANDOM_STATE = 42

Pull data

In [12]:
# PERERA_URL = (
#     "https://raw.githubusercontent.com/"
#     "Paulilein/Perera2018/main/Perera2018_Data_Original.csv"
# )

# df_raw = pd.read_csv(
#     PERERA_URL,
#     sep=";",
#     decimal=",",
#     encoding="utf-8-sig"
# )

# print("Raw shape:", df_raw.shape)
# df_raw.head()

In [13]:
# df_raw.to_csv(
#     DATA_DIR / "perera_pfizer_raw_download.csv",
#     index=False
# )

Renaming amd removing space

In [14]:
# Load the raw data from the local CSV file
df_raw = pd.read_csv(
    DATA_DIR / "perera_pfizer_raw_download.csv"
)

# Remove completely empty "Unnamed" columns
df = df_raw.loc[:, ~df_raw.columns.str.startswith("Unnamed")].copy()

# Remove leading/trailing whitespace from text columns
for col in df.select_dtypes(include="object").columns:
    df[col] = df[col].str.strip()

# # Rename columns to notebook-friendly names #consider removing
# rename_map = {
#     "Reaction_No": "reaction_id",
#     "Reactant_1_Name": "reactant1_name",
#     "Reactant_1_Short_Hand": "reactant1_label",
#     "Reactant_1_eq": "reactant1_eq",
#     "Reactant_1_mmol": "reactant1_mmol",
#     "Reactant_2_Name": "reactant2_label",
#     "Reactant_2_eq": "reactant2_eq",
#     "Catalyst_1_Short_Hand": "catalyst",
#     "Catalyst_1_eq": "catalyst_eq",
#     "Ligand_Short_Hand": "ligand",
#     "Ligand_eq": "ligand_eq",
#     "Reagent_1_Short_Hand": "base",
#     "Reagent_1_eq": "base_eq",
#     "Solvent_1_Short_Hand": "solvent",
#     "Product_Yield_PCT_Area_UV": "yield_pct",
#     "Product_Yield_Mass_Ion_Count": "product_ion_count",
# }

# df = df.rename(columns=rename_map)

# Pandas interprets the literal "None" conditions as missing values.
# Restore those as explicit experimental factor levels.
try:
    df["Ligand_Short_Hand"] = df["Ligand_Short_Hand"].fillna("None")
except KeyError:
    df["ligand"] = df["ligand"].fillna("None")
try:
    df["Reagent_1_Short_Hand"] = df["Reagent_1_Short_Hand"].fillna("None")
except KeyError:
    df["base"] = df["base"].fillna("None")


print(df.shape)
df.head()

(5760, 16)


,Reaction_No,Reactant_1_Name,Reactant_1_Short_Hand,Reactant_1_eq,Reactant_1_mmol,Reactant_2_Name,Reactant_2_eq,Catalyst_1_Short_Hand,Catalyst_1_eq,Ligand_Short_Hand,Ligand_eq,Reagent_1_Short_Hand,Reagent_1_eq,Solvent_1_Short_Hand,Product_Yield_PCT_Area_UV,Product_Yield_Mass_Ion_Count
0,1,6-chloroquinoline,"1a, 6-Cl-Q",1.0,0.0004,"2a, Boronic Acid",1.0,Pd(OAc)2,0.0625,P(tBu)3,0.125,NaOH,2.5,MeCN,4.76,6262.06
1,2,6-chloroquinoline,"1a, 6-Cl-Q",1.0,0.0004,"2a, Boronic Acid",1.0,Pd(OAc)2,0.0625,P(Ph)3,0.125,NaOH,2.5,MeCN,4.12,13245.57
2,3,6-chloroquinoline,"1a, 6-Cl-Q",1.0,0.0004,"2a, Boronic Acid",1.0,Pd(OAc)2,0.0625,AmPhos,0.125,NaOH,2.5,MeCN,2.58,3009.17
3,4,6-chloroquinoline,"1a, 6-Cl-Q",1.0,0.0004,"2a, Boronic Acid",1.0,Pd(OAc)2,0.0625,P(Cy)3,0.125,NaOH,2.5,MeCN,4.44,30860.70
4,5,6-chloroquinoline,"1a, 6-Cl-Q",1.0,0.0004,"2a, Boronic Acid",1.0,Pd(OAc)2,0.0625,P(o-Tol)3,0.125,NaOH,2.5,MeCN,1.95,2486.31


Removing alternative solvent 

In [15]:
print(df["Solvent_1_Short_Hand"].value_counts())

Solvent_1_Short_Hand
MeCN               1440
DMF                1440
THF                1344
MeOH               1344
MeOH/H2O_V2 9:1      96
THF_V2               96
Name: count, dtype: int64


In [16]:
#find rows containing artifacts and drop them
drop_solvent_df = df[df['Solvent_1_Short_Hand'].isin(["MeOH/H2O_V2 9:1","THF_V2"])]
df = df.drop(drop_solvent_df.index, axis=0)

#find rows containing either bidentate or no ligand and drop them
#print(df["Ligand_Short_Hand"].value_counts())
drop_ligand_df = df[df['Ligand_Short_Hand'].isin(["None","dppf","dtbpf","Xantphos"])]
df = df.drop(drop_ligand_df.index, axis=0)
#print(df["Ligand_Short_Hand"].value_counts())

In [17]:
print(df["Reactant_1_Name"].value_counts())

Reactant_1_Name
6-chloroquinoline                         768
6-Bromoquinoline                          768
6-triflatequinoline                       768
6-Iodoquinoline                           768
Potassium quinoline-6-trifluoroborate     256
6-Quinolineboronic acid pinacol ester     256
6-quinoline-boronic acid hydrochloride    128
Name: count, dtype: int64


In [18]:
#df["Reactant_1_Short_Hand"].str.extract(r"^(1[a-z])", expand=False)
df["Reagent_combo"] = (
    df["Reactant_1_Name"] + ", " + df["Reactant_2_Name"] + ", " + df["Ligand_Short_Hand"]
)
print(df["Reagent_combo"].value_counts().shape)
print(df["Reagent_combo"].value_counts())

(120,)
Reagent_combo
6-chloroquinoline, 2a, Boronic Acid, P(tBu)3                      32
6-chloroquinoline, 2a, Boronic Acid, P(Ph)3                       32
6-chloroquinoline, 2a, Boronic Acid, AmPhos                       32
6-chloroquinoline, 2a, Boronic Acid, P(Cy)3                       32
6-chloroquinoline, 2a, Boronic Acid, P(o-Tol)3                    32
                                                                  ..
6-quinoline-boronic acid hydrochloride, 2d, Bromide, P(o-Tol)3    16
6-quinoline-boronic acid hydrochloride, 2d, Bromide, P(Cy)3       16
6-quinoline-boronic acid hydrochloride, 2d, Bromide, AmPhos       16
6-quinoline-boronic acid hydrochloride, 2d, Bromide, P(Ph)3       16
6-quinoline-boronic acid hydrochloride, 2d, Bromide, P(tBu)3      16
Name: count, Length: 120, dtype: int64


In [19]:
print(df["Ligand_Short_Hand"].value_counts())

Ligand_Short_Hand
P(tBu)3        464
P(Ph)3         464
AmPhos         464
P(Cy)3         464
P(o-Tol)3      464
CataCXium A    464
SPhos          464
XPhos          464
Name: count, dtype: int64


In [20]:
audit = pd.Series({
    "rows": len(df),
    "columns": df.shape[1],
    "unique_reaction_ids": df["Reaction_No"].nunique(),
    "duplicate_reaction_ids": df["Reaction_No"].duplicated().sum(),
    "missing_yields": df["Product_Yield_PCT_Area_UV"].isna().sum(),
    "yield_below_0": (df["Product_Yield_PCT_Area_UV"] < 0).sum(),
    "yield_above_100": (df["Product_Yield_PCT_Area_UV"] > 100).sum(),
    "ligand_conditions": df["Ligand_Short_Hand"].nunique(),
    "base_conditions": df["Reagent_1_Short_Hand"].nunique(),
    "solvent_conditions": df["Solvent_1_Short_Hand"].nunique(),
    "Reagent_combinations": df["Reagent_combo"].nunique(),
})

display(audit)

rows                      3712
columns                     17
unique_reaction_ids       3712
duplicate_reaction_ids       0
missing_yields               0
yield_below_0                0
yield_above_100              0
ligand_conditions            8
base_conditions              8
solvent_conditions           4
Reagent_combinations       120
dtype: int64

In [21]:
#isolate group of high quality (1) and low quality (2)
df["reactant1_code"] = (
    df["Reactant_1_Short_Hand"]
    .str.extract(r"^(1[a-z])", expand=False)
)

df["reactant2_code"] = (
    df["Reactant_2_Name"]
    .str.extract(r"^(2[a-z])", expand=False)
)

df["substrate_pair"] = (
    df["reactant1_code"] + "_" + df["reactant2_code"]
)

print("Reactant 1 codes:")
print(df["reactant1_code"].value_counts().sort_index())

print("\nReactant 2 codes:")
print(df["reactant2_code"].value_counts().sort_index())

group1_mask = (
    df["reactant1_code"].isin(["1a", "1b", "1c", "1d"])
    & df["reactant2_code"].isin(["2a", "2b", "2c"])
)

group2_mask = (
    df["reactant1_code"].isin(["1e", "1f", "1g"])
    & df["reactant2_code"].eq("2d")
)

df["reaction_group"] = np.select(
    [group1_mask, group2_mask],
    ["Group1", "Group2"],
    default="Unassigned"
)

print(df["reaction_group"].value_counts())

Reactant 1 codes:
reactant1_code
1a    768
1b    768
1c    768
1d    768
1e    128
1f    256
1g    256
Name: count, dtype: int64

Reactant 2 codes:
reactant2_code
2a    1024
2b    1024
2c    1024
2d     640
Name: count, dtype: int64
reaction_group
Group1    3072
Group2     640
Name: count, dtype: int64


In [22]:
#isolate high qaulity
df_group1 = df[df["reaction_group"] == "Group1"].copy()

print(df_group1.shape)

(3072, 21)


Pull Kraken data

In [23]:
# KRAKEN_IDENTIFIERS_URL = (
#     "https://raw.githubusercontent.com/"
#     "doyle-lab-ucla/kraken_utils/main/identifiers.csv"
# )

# KRAKEN_FEATURES_URL = (
#     "https://raw.githubusercontent.com/"
#     "doyle-lab-ucla/kraken_utils/main/kraken_features_only.csv"
# )

# kraken_ids = pd.read_csv(KRAKEN_IDENTIFIERS_URL)

# kraken_features = pd.read_csv(KRAKEN_FEATURES_URL)

# kraken_ids["id"] = pd.to_numeric(
#     kraken_ids["id"]
# ).astype("Int64")

# kraken_features["id"] = pd.to_numeric(
#     kraken_features["id"]
# ).astype(int)

# print("Identifier table:", kraken_ids.shape)
# print("Descriptor table:", kraken_features.shape)
# kraken_ids.to_csv(
#     DATA_DIR / "kraken_identifiers.csv",
#     index=False
# )

# kraken_features.to_csv(
#     DATA_DIR / "kraken_features_only.csv",
#     index=False
# )

In [24]:
# local data
kraken_ids = pd.read_csv(DATA_DIR / "kraken_identifiers.csv")

kraken_features = pd.read_csv(DATA_DIR / "kraken_features_only.csv")

In [25]:
# ID matching to kraken
PERERA_TO_KRAKEN = {
    "XPhos": 1,
    "SPhos": 3,
    "P(tBu)3": 8,
    "P(o-Tol)3": 9,
    "CataCXium A": 10,
    "P(Cy)3": 11,
    "P(Ph)3": 17,
    "AmPhos": 216,
}

In [35]:
# check ID and name matching
mapping_df = pd.DataFrame({
    "perera_ligand": list(PERERA_TO_KRAKEN.keys()),
    "kraken_id": list(PERERA_TO_KRAKEN.values())
})

mapping_check = mapping_df.merge(
    kraken_ids[["id", "ligand", "can_smiles"]]
    .rename(columns={
        "id": "kraken_id",
        "ligand": "kraken_name"
    }),
    on="kraken_id",
    how="left",
    validate="one_to_one"
)

display(mapping_check)

,perera_ligand,kraken_id,kraken_name,can_smiles
0,XPhos,1,Xphos,CC(C)c1cc(C(C)C)c(-c2ccccc2P(C2CCCCC2)C2CCCCC2...
1,SPhos,3,SPhos,COc1cccc(OC)c1-c1ccccc1P(C1CCCCC1)C1CCCCC1
2,P(tBu)3,8,PtBu3,CC(C)(C)P(C(C)(C)C)C(C)(C)C
3,P(o-Tol)3,9,PoTol3,Cc1ccccc1P(c1ccccc1C)c1ccccc1C
4,CataCXium A,10,"PAd2nBu, CataCXium A",CCCCP(C12CC3CC(C2)CC(C1)C3)C12CC3CC(C2)CC(C1)C3
5,P(Cy)3,11,PCy3,C1CCC(P(C2CCCCC2)C2CCCCC2)CC1
6,P(Ph)3,17,PPh3,c1ccc(P(c2ccccc2)c2ccccc2)cc1
7,AmPhos,216,Ata Phos / AmPhos,CN(C)c1ccc(P(C(C)(C)C)C(C)(C)C)cc1


Kraken feature extraction and conversions

In [ ]:
# wanted_raw_cols = [
#     "id",
#     "vmin_vmin_boltz",
#     "fmo_e_homo_boltz",
#     "fmo_e_lumo_boltz",
#     "dipolemoment_boltz",
#     "vbur_vbur_boltz",
#     "vbur_vbur_min",
#     "vbur_vbur_max",
#     "vbur_vbur_delta",
# ]
# #kraken_features[wanted_raw_cols].head()

# transform vburr to % vburr using 3.5 A radius
sph_vol = 4/3*np.pi*(3.5**3)
kraken_features["%vbur_boltz"] = kraken_features["vbur_vbur_boltz"] / sph_vol *100
kraken_features["%vbur_min"] = kraken_features["vbur_vbur_min"] / sph_vol
kraken_features["%vbur_max"] = kraken_features["vbur_vbur_max"] / sph_vol
kraken_features["%vbur_delta"] = kraken_features["vbur_vbur_delta"] / sph_vol


#conversion and calculation for electronic features
Hartree_to_eV = 27.211386245981    #NIST value
kraken_features["fmo_e_homo_boltz"] = kraken_features["fmo_e_homo_boltz"] * Hartree_to_eV
kraken_features["fmo_e_lumo_boltz"] = kraken_features["fmo_e_lumo_boltz"] * Hartree_to_eV
kraken_features["delta_band_gap"] = kraken_features["fmo_e_homo_boltz"] - kraken_features["fmo_e_lumo_boltz"] 



wanted_updated_cols =[
    "id",
    "vmin_vmin_boltz",
    "fmo_e_homo_boltz",
    "fmo_e_lumo_boltz",
    "dipolemoment_boltz",
    "%vbur_boltz",
    "%vbur_min",
    "%vbur_max",
    "%vbur_delta",
    "delta_band_gap",
]
kraken_features[wanted_updated_cols]


,id,vmin_vmin_boltz,fmo_e_homo_boltz,fmo_e_lumo_boltz,dipolemoment_boltz,%vbur_boltz,%vbur_min,%vbur_max,%vbur_delta,delta_band_gap
0,1,-0.0617,-5.9387,-0.6833,1.2581,0.5878,0.3144,0.7123,0.3980,-5.2554
1,2,-0.0637,-5.6140,-0.6334,0.8172,0.6052,0.3259,0.6946,0.3687,-4.9806
2,3,-0.0663,-5.8048,-0.4605,0.9112,0.5629,0.3167,0.5828,0.2661,-5.3443
3,4,-0.0673,-5.7572,-0.3756,1.0272,0.6003,0.3086,0.7265,0.4179,-5.3816
4,5,-0.0614,-5.9550,-0.8203,1.0883,0.4752,0.3176,0.6208,0.3032,-5.1347
...,...,...,...,...,...,...,...,...,...,...
1218,1522,-0.0435,-6.0534,0.1726,1.2052,0.3105,0.2856,0.3109,0.0253,-6.2260
1219,1523,-0.0253,-6.6838,-0.3675,1.2182,0.2707,0.2406,0.3037,0.0631,-6.3163
1220,1524,-0.0408,-6.4154,-0.9970,2.5124,0.2643,0.2643,0.2643,0.0000,-5.4183
1221,1525,-0.0051,-7.7039,-0.8339,1.9483,0.2155,0.2084,0.2352,0.0268,-6.8700


In [36]:
#Specific ligand isolation
ligand_desc = (
    kraken_features[
        kraken_features["id"].isin(PERERA_TO_KRAKEN.values())
    ][wanted_updated_cols]
    .copy()
    .rename(columns={"id": "kraken_id"})
)
ligand_desc["ligand"] = (
    ligand_desc["kraken_id"]
    .map({kraken_id: ligand for ligand, kraken_id in PERERA_TO_KRAKEN.items()})
)

# Move the ligand label to the front without overwriting another column
#ligand_desc.insert(0, "ligand", ligand_desc.pop("ligand"))

ligand_desc.head(9)


,kraken_id,vmin_vmin_boltz,fmo_e_homo_boltz,fmo_e_lumo_boltz,dipolemoment_boltz,%vbur_boltz,%vbur_min,%vbur_max,%vbur_delta,delta_band_gap,ligand
0,1,-0.0617,-5.9387,-0.6833,1.2581,0.5878,0.3144,0.7123,0.3980,-5.2554,XPhos
2,3,-0.0663,-5.8048,-0.4605,0.9112,0.5629,0.3167,0.5828,0.2661,-5.3443,SPhos
7,8,-0.0672,-5.9016,0.9799,1.0833,0.3626,0.3626,0.3626,0.0000,-6.8815,P(tBu)3
8,9,-0.0435,-6.0569,-0.7646,0.4321,0.3996,0.3438,0.3997,0.0558,-5.2923,P(o-Tol)3
9,10,-0.0673,-5.9055,0.5655,1.2028,0.3580,0.3281,0.4201,0.0920,-6.4710,CataCXium A
10,11,-0.0668,-5.9902,0.8726,1.2805,0.3203,0.3020,0.3983,0.0962,-6.8628,P(Cy)3
16,17,-0.0482,-6.2148,-0.8025,1.4605,0.2822,0.2822,0.2822,0.0000,-5.4123,P(Ph)3
211,216,-0.0702,-5.5968,-0.2337,3.2409,0.3517,0.3517,0.3517,0.0000,-5.3631,AmPhos


In [37]:
df_group1.head()

,Reaction_No,Reactant_1_Name,Reactant_1_Short_Hand,Reactant_1_eq,Reactant_1_mmol,Reactant_2_Name,Reactant_2_eq,Catalyst_1_Short_Hand,Catalyst_1_eq,Ligand_Short_Hand,Ligand_eq,Reagent_1_Short_Hand,Reagent_1_eq,Solvent_1_Short_Hand,Product_Yield_PCT_Area_UV,Product_Yield_Mass_Ion_Count,Reagent_combo,reactant1_code,reactant2_code,substrate_pair,reaction_group,kraken_id
0,1,6-chloroquinoline,"1a, 6-Cl-Q",1.0,0.0004,"2a, Boronic Acid",1.0,Pd(OAc)2,0.0625,P(tBu)3,0.125,NaOH,2.5,MeCN,4.76,6262.06,"6-chloroquinoline, 2a, Boronic Acid, P(tBu)3",1a,2a,1a_2a,Group1,8
1,2,6-chloroquinoline,"1a, 6-Cl-Q",1.0,0.0004,"2a, Boronic Acid",1.0,Pd(OAc)2,0.0625,P(Ph)3,0.125,NaOH,2.5,MeCN,4.12,13245.57,"6-chloroquinoline, 2a, Boronic Acid, P(Ph)3",1a,2a,1a_2a,Group1,17
2,3,6-chloroquinoline,"1a, 6-Cl-Q",1.0,0.0004,"2a, Boronic Acid",1.0,Pd(OAc)2,0.0625,AmPhos,0.125,NaOH,2.5,MeCN,2.58,3009.17,"6-chloroquinoline, 2a, Boronic Acid, AmPhos",1a,2a,1a_2a,Group1,216
3,4,6-chloroquinoline,"1a, 6-Cl-Q",1.0,0.0004,"2a, Boronic Acid",1.0,Pd(OAc)2,0.0625,P(Cy)3,0.125,NaOH,2.5,MeCN,4.44,30860.70,"6-chloroquinoline, 2a, Boronic Acid, P(Cy)3",1a,2a,1a_2a,Group1,11
4,5,6-chloroquinoline,"1a, 6-Cl-Q",1.0,0.0004,"2a, Boronic Acid",1.0,Pd(OAc)2,0.0625,P(o-Tol)3,0.125,NaOH,2.5,MeCN,1.95,2486.31,"6-chloroquinoline, 2a, Boronic Acid, P(o-Tol)3",1a,2a,1a_2a,Group1,9


Focused  model (select features)

In [38]:
df_group1["kraken_id"] = (
    df_group1["Ligand_Short_Hand"]
    .map(PERERA_TO_KRAKEN)
    .astype("Int64")
)

model_df = df_group1.merge(
    ligand_desc,
    on=["kraken_id"],
    how="left",
    validate="many_to_one"
)

print("Modeling table shape:", model_df.shape)
# Move the ligand label to the front without overwriting another column
model_df.insert(0, "ligand", model_df.pop("ligand"))
#drop Ligand_Short_Hand 
model_df.drop(columns=["Ligand_Short_Hand"], inplace=True)
model_df.head(100)

Modeling table shape: (3072, 32)


,ligand,Reaction_No,Reactant_1_Name,Reactant_1_Short_Hand,Reactant_1_eq,Reactant_1_mmol,Reactant_2_Name,Reactant_2_eq,Catalyst_1_Short_Hand,Catalyst_1_eq,Ligand_eq,Reagent_1_Short_Hand,Reagent_1_eq,Solvent_1_Short_Hand,Product_Yield_PCT_Area_UV,Product_Yield_Mass_Ion_Count,Reagent_combo,reactant1_code,reactant2_code,substrate_pair,reaction_group,kraken_id,vmin_vmin_boltz,fmo_e_homo_boltz,fmo_e_lumo_boltz,dipolemoment_boltz,%vbur_boltz,%vbur_min,%vbur_max,%vbur_delta,delta_band_gap
0,P(tBu)3,1,6-chloroquinoline,"1a, 6-Cl-Q",1.0,0.0004,"2a, Boronic Acid",1.0,Pd(OAc)2,0.0625,0.125,NaOH,2.5,MeCN,4.76,6262.06,"6-chloroquinoline, 2a, Boronic Acid, P(tBu)3",1a,2a,1a_2a,Group1,8,-0.0672,-5.9016,0.9799,1.0833,0.3626,0.3626,0.3626,0.0000,-6.8815
1,P(Ph)3,2,6-chloroquinoline,"1a, 6-Cl-Q",1.0,0.0004,"2a, Boronic Acid",1.0,Pd(OAc)2,0.0625,0.125,NaOH,2.5,MeCN,4.12,13245.57,"6-chloroquinoline, 2a, Boronic Acid, P(Ph)3",1a,2a,1a_2a,Group1,17,-0.0482,-6.2148,-0.8025,1.4605,0.2822,0.2822,0.2822,0.0000,-5.4123
2,AmPhos,3,6-chloroquinoline,"1a, 6-Cl-Q",1.0,0.0004,"2a, Boronic Acid",1.0,Pd(OAc)2,0.0625,0.125,NaOH,2.5,MeCN,2.58,3009.17,"6-chloroquinoline, 2a, Boronic Acid, AmPhos",1a,2a,1a_2a,Group1,216,-0.0702,-5.5968,-0.2337,3.2409,0.3517,0.3517,0.3517,0.0000,-5.3631
3,P(Cy)3,4,6-chloroquinoline,"1a, 6-Cl-Q",1.0,0.0004,"2a, Boronic Acid",1.0,Pd(OAc)2,0.0625,0.125,NaOH,2.5,MeCN,4.44,30860.70,"6-chloroquinoline, 2a, Boronic Acid, P(Cy)3",1a,2a,1a_2a,Group1,11,-0.0668,-5.9902,0.8726,1.2805,0.3203,0.3020,0.3983,0.0962,-6.8628
4,P(o-Tol)3,5,6-chloroquinoline,"1a, 6-Cl-Q",1.0,0.0004,"2a, Boronic Acid",1.0,Pd(OAc)2,0.0625,0.125,NaOH,2.5,MeCN,1.95,2486.31,"6-chloroquinoline, 2a, Boronic Acid, P(o-Tol)3",1a,2a,1a_2a,Group1,9,-0.0435,-6.0569,-0.7646,0.4321,0.3996,0.3438,0.3997,0.0558,-5.2923
5,CataCXium A,6,6-chloroquinoline,"1a, 6-Cl-Q",1.0,0.0004,"2a, Boronic Acid",1.0,Pd(OAc)2,0.0625,0.125,NaOH,2.5,MeCN,3.25,9592.45,"6-chloroquinoline, 2a, Boronic Acid, CataCXium A",1a,2a,1a_2a,Group1,10,-0.0673,-5.9055,0.5655,1.2028,0.3580,0.3281,0.4201,0.0920,-6.4710
6,SPhos,7,6-chloroquinoline,"1a, 6-Cl-Q",1.0,0.0004,"2a, Boronic Acid",1.0,Pd(OAc)2,0.0625,0.125,NaOH,2.5,MeCN,86.26,108343.14,"6-chloroquinoline, 2a, Boronic Acid, SPhos",1a,2a,1a_2a,Group1,3,-0.0663,-5.8048,-0.4605,0.9112,0.5629,0.3167,0.5828,0.2661,-5.3443
7,XPhos,9,6-chloroquinoline,"1a, 6-Cl-Q",1.0,0.0004,"2a, Boronic Acid",1.0,Pd(OAc)2,0.0625,0.125,NaOH,2.5,MeCN,93.67,27239.20,"6-chloroquinoline, 2a, Boronic Acid, XPhos",1a,2a,1a_2a,Group1,1,-0.0617,-5.9387,-0.6833,1.2581,0.5878,0.3144,0.7123,0.3980,-5.2554
8,P(tBu)3,13,6-chloroquinoline,"1a, 6-Cl-Q",1.0,0.0004,"2a, Boronic Acid",1.0,Pd(OAc)2,0.0625,0.125,NaHCO3,2.5,MeCN,3.16,4185.92,"6-chloroquinoline, 2a, Boronic Acid, P(tBu)3",1a,2a,1a_2a,Group1,8,-0.0672,-5.9016,0.9799,1.0833,0.3626,0.3626,0.3626,0.0000,-6.8815
9,P(Ph)3,14,6-chloroquinoline,"1a, 6-Cl-Q",1.0,0.0004,"2a, Boronic Acid",1.0,Pd(OAc)2,0.0625,0.125,NaHCO3,2.5,MeCN,4.75,12755.63,"6-chloroquinoline, 2a, Boronic Acid, P(Ph)3",1a,2a,1a_2a,Group1,17,-0.0482,-6.2148,-0.8025,1.4605,0.2822,0.2822,0.2822,0.0000,-5.4123


In [39]:
# descriptor_check_cols = [
#     "%vbur_min",
#     "%vbur_boltz",
#     "%vbur_delta",
#     "vmin_vmin_boltz",
#     "delta_band_gap",
#     "dipolemoment_boltz",
# ]

# print(
#     model_df[descriptor_check_cols]
#     .isna()
#     .sum()
# )

Full Feature Model

In [40]:
kraken_full = kraken_features.rename(
    columns={"id": "kraken_id"}
).copy()

model_df_full = df_group1.merge(
    kraken_full,
    on="kraken_id",
    how="left",
    validate="many_to_one"
)

print(model_df_full.shape)

(3072, 217)


Save datasets

In [41]:
model_df.to_csv(
    DATA_DIR / "perera_pfizer_group1_kraken_interpretable.csv",
    index=False
)

model_df_full.to_csv(
    DATA_DIR / "perera_pfizer_group1_kraken_full.csv",
    index=False
)